In [1]:
'''
In this set of experiments, we identify noisy nodes using NLD entropy. The idea is to identify nodes removing which reduces the NLD entropy of its
neighbors

As not all labels are available, we train a GCN to get pseudo labels.

Psuedo labels do not overwrite the existing labels, and are only used for labels for which the labels are not originally available

'''


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.datasets import Amazon
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.transforms import RandomNodeSplit
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_remaining_self_loops
from tqdm import tqdm
import numpy as np
import copy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [2]:
'''Utility Functions'''
def corrupt_mask(labels, mask, noise_rate, num_classes):
    """Applies Symmetric Label Noise to a specific mask."""
    noisy_labels = labels.clone()
    indices = torch.nonzero(mask).squeeze()
    num_target = indices.size(0)
    num_noisy = int(noise_rate * num_target)
    
    noisy_idx = torch.randperm(num_target)[:num_noisy]
    corrupted_nodes = indices[noisy_idx]
    
    random_classes = torch.randint(0, num_classes, (num_noisy,), device=labels.device)
    noisy_labels[corrupted_nodes] = random_classes
    return noisy_labels

def corrupt_mask_pairwise(labels, mask, noise_rate, num_classes):
    """Applies Pairwise Label Noise to a specific mask.
    Transitions class i to (i + 1) % num_classes.
    """
    noisy_labels=labels.clone()
    indices = torch.nonzero(mask).squeeze()
    num_target = indices.size(0)
    num_noisy = int(noise_rate * num_target)
    
    # Randomly select nodes to corrupt
    noisy_idx = torch.randperm(num_target)[:num_noisy]
    corrupted_nodes = indices[noisy_idx]
    
    # Apply the pairwise rotation: class i -> class i+1
    old_classes = noisy_labels[corrupted_nodes]
    new_classes = (old_classes + 1) % num_classes
    
    noisy_labels[corrupted_nodes] = new_classes
    return noisy_labels


def calculate_entropy(counts):
    """Calculates Shannon entropy given raw class counts."""
    degrees = counts.sum(dim=1, keepdim=True)
    degrees[degrees == 0] = 1.0
    probs = counts / degrees
    probs_safe = torch.where(probs > 0, probs, torch.tensor(1e-9, device=probs.device))
    return -torch.sum(probs * torch.log(probs_safe), dim=1)

def identify_noisy_nodes_realistic(edge_index, labels, num_nodes, num_classes, sum_threshold=0.0, vote_threshold=0.5, self_entropy_threshold=1.2):
    """Leave-one-out entropy impact analysis returning both sum and vote methods."""
    Y = F.one_hot(labels, num_classes=num_classes).float()
    val = torch.ones(edge_index.size(1), device=device)
    adj = torch.sparse_coo_tensor(edge_index, val, (num_nodes, num_nodes))
    
    orig_counts = torch.sparse.mm(adj, Y)
    orig_entropy = calculate_entropy(orig_counts)
    
    entropy_diff_sum = torch.zeros(num_nodes, device=device)
    positive_diff_votes = torch.zeros(num_nodes, device=device)
    degree_counts = torch.zeros(num_nodes, device=device)

    v_nodes = edge_index[0]
    u_nodes = edge_index[1]
    
    u_counts = orig_counts[u_nodes]
    v_labels = labels[v_nodes]
    v_labels_one_hot = F.one_hot(v_labels, num_classes=num_classes).float()
    
    new_u_counts = torch.clamp(u_counts - v_labels_one_hot, min=0.0)
    new_u_entropy = calculate_entropy(new_u_counts)
    
    diff = orig_entropy[u_nodes] - new_u_entropy
    positive_diff = (diff > 0).float()
    
    entropy_diff_sum.scatter_add_(0, v_nodes, diff)
    positive_diff_votes.scatter_add_(0, v_nodes, positive_diff)
    
    ones = torch.ones_like(v_nodes, dtype=torch.float)
    degree_counts.scatter_add_(0, v_nodes, ones)
    
    degree_counts[degree_counts == 0] = 1.0 
    vote_fraction = positive_diff_votes / degree_counts
    
    is_low_self_entropy = orig_entropy < self_entropy_threshold
    
    pred_noisy_sum = (entropy_diff_sum > sum_threshold) & is_low_self_entropy
    pred_noisy_vote = (vote_fraction > vote_threshold) & is_low_self_entropy
    
    return pred_noisy_sum, pred_noisy_vote

In [3]:
'''Model Definition'''
class GCNNet(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super(GCNNet, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h_relu = F.relu(h)
        h_drop = F.dropout(h_relu, p=0.5, training=self.training)
        out = self.conv2(h_drop, edge_index)
        return F.log_softmax(out, dim=1)



In [4]:
'''Training loop'''
def train_and_evaluate(model, x, edge_index, target_labels, train_mask, val_mask, test_mask, true_labels, patience=40):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(1, 1000):
        model.train()
        opt.zero_grad()
        out = model(x, edge_index)
        loss = F.nll_loss(out[train_mask], target_labels[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_out = model(x, edge_index)
            # Validation loss is calculated against the target_labels 
            # In a strict realistic setting, val labels are clean.
            val_loss = F.nll_loss(val_out[val_mask], true_labels[val_mask])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out = model(x, edge_index)
        pred = out.argmax(dim=1)
        test_acc = (pred[test_mask] == true_labels[test_mask]).sum().item() / test_mask.sum().item()

    return test_acc, pred

### Results in presence of Symmetric Label Noise

In [5]:


'''Execution'''
noise_rates = [0,0.1, 0.2, 0.3, 0.4, 0.5]
num_runs = 5
entropy_shield_limit = 1.2 

for noise_rate in noise_rates:
    acc_clean_sum = []
    acc_clean_vote = []
    
    print(f"\n--- Training Set Noise level: {noise_rate} ---")
    
    for run in range(num_runs):
        dataset = Amazon(root='/tmp', name='Photo',transform=NormalizeFeatures())
        split=RandomNodeSplit(split="random",num_train_per_class=54,num_val=50,num_test=0.8)
        data = split(dataset[0]).to(device)
        data.edge_index, _ = add_remaining_self_loops(data.edge_index, num_nodes=data.num_nodes)

        num_classes = dataset.num_classes
        num_features = dataset.num_node_features
        # Inject noise ONLY into the training mask
        noisy_y = data.y.clone()
        noisy_y = corrupt_mask(noisy_y, data.train_mask, noise_rate, num_classes)
        noisy_y = corrupt_mask(noisy_y, data.val_mask, noise_rate, num_classes)

        
        # 1. Baseline Model
        baseline_model = GCNNet(num_features, 64, num_classes).to(device)
        baseline_test_acc, pred_baseline = train_and_evaluate(
            baseline_model, data.x, data.edge_index, noisy_y, 
            data.train_mask, data.val_mask, data.test_mask, data.y
        )
 

        # 2. Construct Fusion Labels (Noisy Train + Pseudo Val/Test)
        working_labels = pred_baseline.clone()
        working_labels[data.train_mask] = noisy_y[data.train_mask]

        # 3. Identify Noisy Nodes
        pred_noisy_sum, pred_noisy_vote = identify_noisy_nodes_realistic(
            data.edge_index, working_labels, data.num_nodes, num_classes,
            sum_threshold=0.0, vote_threshold=0.5, self_entropy_threshold=entropy_shield_limit
        )

        # 4. Clean Graph & Train Sum Variant
        clean_train_mask_sum = data.train_mask & ~pred_noisy_sum
        sum_model = GCNNet(num_features, 64, num_classes).to(device)
        sum_test_acc, _ = train_and_evaluate(
            sum_model, data.x, data.edge_index, noisy_y, 
            clean_train_mask_sum, data.val_mask, data.test_mask, data.y
        )
        acc_clean_sum.append(sum_test_acc)

        # 5. Clean Graph & Train Vote Variant
        clean_train_mask_vote = data.train_mask & ~pred_noisy_vote
        vote_model = GCNNet(num_features, 64, num_classes).to(device)
        vote_test_acc, _ = train_and_evaluate(
            vote_model, data.x, data.edge_index, noisy_y, 
            clean_train_mask_vote, data.val_mask, data.test_mask, data.y
        )
        acc_clean_vote.append(vote_test_acc)

    print(f"  NLD Clean (Sum)  -> Accuracy: {np.mean(acc_clean_sum):.4f} ± {np.std(acc_clean_sum):.4f}")
    print(f"  NLD Clean (Vote) -> Accuracy: {np.mean(acc_clean_vote):.4f} ± {np.std(acc_clean_vote):.4f}")


--- Training Set Noise level: 0 ---


/tmp/ipykernel_2429770/3329018769.py:49: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(edge_index, val, (num_nodes, num_nodes))


  NLD Clean (Sum)  -> Accuracy: 0.9157 ± 0.0079
  NLD Clean (Vote) -> Accuracy: 0.9191 ± 0.0075

--- Training Set Noise level: 0.1 ---
  NLD Clean (Sum)  -> Accuracy: 0.9166 ± 0.0021
  NLD Clean (Vote) -> Accuracy: 0.9203 ± 0.0024

--- Training Set Noise level: 0.2 ---
  NLD Clean (Sum)  -> Accuracy: 0.9012 ± 0.0097
  NLD Clean (Vote) -> Accuracy: 0.9005 ± 0.0079

--- Training Set Noise level: 0.3 ---
  NLD Clean (Sum)  -> Accuracy: 0.8890 ± 0.0130
  NLD Clean (Vote) -> Accuracy: 0.8996 ± 0.0084

--- Training Set Noise level: 0.4 ---
  NLD Clean (Sum)  -> Accuracy: 0.8955 ± 0.0142
  NLD Clean (Vote) -> Accuracy: 0.8940 ± 0.0178

--- Training Set Noise level: 0.5 ---
  NLD Clean (Sum)  -> Accuracy: 0.8721 ± 0.0159
  NLD Clean (Vote) -> Accuracy: 0.8719 ± 0.0078


### Results in presence of Pairwise Noise

In [8]:
'''Psuedo labels do not overwrite the existing labels
For pairwise noise
'''




'''Execution'''

noise_rates = [0.1, 0.2, 0.3, 0.4]
num_runs = 5
entropy_shield_limit = 1.2 

for noise_rate in noise_rates:
    acc_clean_sum = []
    acc_clean_vote = []
    
    print(f"\n--- Training Set Noise level: {noise_rate} ---")
    
    for run in range(num_runs):

        dataset = Amazon(root='/tmp', name='Photo',transform=NormalizeFeatures())
        split=RandomNodeSplit(split="random",num_train_per_class=54,num_val=50,num_test=0.8)
        data = split(dataset[0]).to(device)
        data.edge_index, _ = add_remaining_self_loops(data.edge_index, num_nodes=data.num_nodes)

        num_classes = dataset.num_classes
        num_features = dataset.num_node_features

        noisy_y = data.y.clone()
        noisy_y = corrupt_mask_pairwise(noisy_y, data.train_mask, noise_rate, num_classes)
        noisy_y = corrupt_mask_pairwise(noisy_y, data.val_mask, noise_rate, num_classes)

        
        
        # 1. Baseline Model
        baseline_model = GCNNet(num_features, 64, num_classes).to(device)
        baseline_test_acc, pred_baseline = train_and_evaluate(
            baseline_model, data.x, data.edge_index, noisy_y, 
            data.train_mask, data.val_mask, data.test_mask, data.y
        )
        # 2. Construct Fusion Labels (Noisy Train + Pseudo Val/Test)
        working_labels = pred_baseline.clone()
        working_labels[data.train_mask] = noisy_y[data.train_mask]

        # 3. Identify Noisy Nodes
        pred_noisy_sum, pred_noisy_vote = identify_noisy_nodes_realistic(
            data.edge_index, working_labels, data.num_nodes, num_classes,
            sum_threshold=0.0, vote_threshold=0.5, self_entropy_threshold=entropy_shield_limit
        )

        # 4. Clean Graph & Train Sum Variant
        clean_train_mask_sum = data.train_mask & ~pred_noisy_sum
        sum_model = GCNNet(num_features, 64, num_classes).to(device)
        sum_test_acc, _ = train_and_evaluate(
            sum_model, data.x, data.edge_index, noisy_y, 
            clean_train_mask_sum, data.val_mask, data.test_mask, data.y
        )
        acc_clean_sum.append(sum_test_acc)

        # 5. Clean Graph & Train Vote Variant
        clean_train_mask_vote = data.train_mask & ~pred_noisy_vote
        vote_model = GCNNet(num_features, 64, num_classes).to(device)
        vote_test_acc, _ = train_and_evaluate(
            vote_model, data.x, data.edge_index, noisy_y, 
            clean_train_mask_vote, data.val_mask, data.test_mask, data.y
        )
        acc_clean_vote.append(vote_test_acc)

    print(f"  NLD Clean (Sum)  -> Accuracy: {np.mean(acc_clean_sum):.4f} ± {np.std(acc_clean_sum):.4f}")
    print(f"  NLD Clean (Vote) -> Accuracy: {np.mean(acc_clean_vote):.4f} ± {np.std(acc_clean_vote):.4f}")


--- Training Set Noise level: 0.1 ---
  NLD Clean (Sum)  -> Accuracy: 0.9079 ± 0.0090
  NLD Clean (Vote) -> Accuracy: 0.9151 ± 0.0068

--- Training Set Noise level: 0.2 ---
  NLD Clean (Sum)  -> Accuracy: 0.9088 ± 0.0069
  NLD Clean (Vote) -> Accuracy: 0.9139 ± 0.0051

--- Training Set Noise level: 0.3 ---
  NLD Clean (Sum)  -> Accuracy: 0.8644 ± 0.0134
  NLD Clean (Vote) -> Accuracy: 0.8738 ± 0.0080

--- Training Set Noise level: 0.4 ---
  NLD Clean (Sum)  -> Accuracy: 0.7772 ± 0.0352
  NLD Clean (Vote) -> Accuracy: 0.7792 ± 0.0281
